# Cunningham Essays — Q&A Dataset Generation

**Source:** `Essays-of-Warren-Buffett-Lessons-for-Corporate-America_Cunningham.pdf` (1.2 MB, 219 pages)  
**Content:** Warren Buffett's shareholder letter excerpts (1977–2012), thematically organized by Lawrence Cunningham across 5 chapters and 32 sub-sections. Financial tables and third-party reports have been removed by Cunningham — the text is pure Buffett prose. Zero content overlap with the partnership letters (1957–1970).

**Key advantage:** Section headers map directly to our 6 assignment labels, enabling structural pre-labeling without LLM classification calls for ~85% of chunks.

In [1]:
import sys, re, statistics, importlib
from pathlib import Path
from collections import Counter

sys.path.insert(0, "..")
import pipeline.core
importlib.reload(pipeline.core)
from pipeline.core import (
    Chunk, QAPair, LABELS, extract_text,
    classify_chunks, generate_all, score_all,
    filter_by_quality, coverage_audit,
    export_csv, export_detailed,
    save_checkpoint, load_checkpoint,
)
import fitz

## Stage 1: Chunking

The document contains 32 named sub-sections (A. through J. under 5 Roman numeral chapters). Boundary detection matches `A. <Section Name>` patterns against a known section list to avoid false positives from internal sub-points (e.g., Buffett's satirical accounting letter contains its own A–D items that are not real section boundaries).

The body text starts at the second occurrence of "I. CORPORATE GOVERNANCE" — the first is in the Table of Contents. Cunningham's third-person editorial paragraphs (identified by phrases like "Buffett argues" without any first-person voice) are filtered before chunking.

Validated in `test_cunningham_chunking_v2.py`.

In [2]:
SECTION_TO_LABEL = {
    "Owner-Related Business Principles": "Strategy Development",
    "Boards and Managers": "Strategy Development",
    "Anxieties of Plant Closings": "Adaptability",
    "Owner-Based Approach to Corporate Charity": "Personal Life",
    "Principled Approach to Executive Pay": "Strategy Development",
    "Mr. Market": "Psychology",
    "Arbitrage": "Timing",
    "Debunking Standard Dogma": "Strategy Development",
    "Intelligent Investing": "Strategy Development",
    "Cigar Butts and the Institutional Imperative": "Strategy Development",
    "Junk Bonds": "Risk Management",
    "Zero-Coupon Bonds": "Risk Management",
    "Preferred Stock": "Risk Management",
    "Bane of Trading": "Psychology",
    "Attracting the Right Sort of Investor": "Psychology",
    "Dividend Policy": "Strategy Development",
    "Stock Splits and Trading Activity": "Timing",
    "Shareholder Strategies": "Strategy Development",
    "Berkshire's Recapitalization": "Strategy Development",
    "Bad Motives and High Prices": "Risk Management",
    "Sensible Stock Repurchases": "Timing",
    "Leveraged Buyouts": "Risk Management",
    "Sound Acquisition Policies": "Strategy Development",
    "On Selling One's Business": "Strategy Development",
    "Satire on Accounting Shenanigans": "Adaptability",
    "Look-Through Earnings": "Strategy Development",
    "Economic Goodwill Versus Accounting Goodwill": "Strategy Development",
    "Economic Goodwill": "Strategy Development",
    "Owner Earnings and the Cash Flow Fallacy": "Strategy Development",
    "Owner Earnings": "Strategy Development",
    "Intrinsic Value, Book Value, and Market Price": "Timing",
    "Intrinsic Value": "Timing",
    "Segment Data and Consolidation": "Strategy Development",
    "Segment Data": "Strategy Development",
    "Deferred Taxes": "Risk Management",
    "Retiree Benefits and Stock Options": "Risk Management",
    "Retiree Benefits": "Risk Management",
    "Distribution of the Corporate Tax Burden": "Adaptability",
    "Taxation and Investment Philosophy": "Strategy Development",
}

KNOWN_SECTIONS = [
    "Owner-Related Business Principles", "Boards and Managers",
    "Anxieties of Plant Closings", "Owner-Based Approach to Corporate Charity",
    "Principled Approach to Executive Pay", "Mr. Market", "Arbitrage",
    "Debunking Standard Dogma", "Intelligent Investing",
    "Cigar Butts and the Institutional Imperative", "Junk Bonds",
    "Zero-Coupon Bonds", "Preferred Stock", "Bane of Trading",
    "Attracting the Right Sort of Investor", "Dividend Policy",
    "Stock Splits and Trading Activity", "Shareholder Strategies",
    "Berkshire's Recapitalization", "Bad Motives and High Prices",
    "Sensible Stock Repurchases", "Leveraged Buyouts",
    "Sound Acquisition Policies", "On Selling One's Business",
    "Satire on Accounting Shenanigans", "Look-Through Earnings",
    "Economic Goodwill", "Owner Earnings", "Intrinsic Value",
    "Segment Data", "Deferred Taxes", "Retiree Benefits",
    "Distribution of the Corporate Tax Burden",
    "Taxation and Investment Philosophy",
]


def extract_essays_text(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = "".join(pg.get_text() for pg in doc)
    doc.close()
    # Skip TOC + Cunningham intro — body starts at second "I. CORPORATE GOVERNANCE"
    matches = list(re.finditer(r'I\.\s*\n\s*CORPORATE GOVERNANCE', full_text))
    if len(matches) >= 2:
        return full_text[matches[1].start():]
    prologue = re.search(r'PROLOGUE', full_text)
    return full_text[prologue.start():] if prologue else full_text


def find_section_boundaries(text):
    boundaries = []
    pattern = re.compile(r'^([A-J])\.\s*\n?\s*([A-Z][a-zA-Z\s\-,\'\(\)&]+)', re.MULTILINE)
    for m in pattern.finditer(text):
        raw_name = m.group(2).strip().split('\n')[0].strip()
        matched_known = False
        for known in KNOWN_SECTIONS:
            if known.lower() in raw_name.lower() or raw_name.lower() in known.lower():
                matched_known = True
                raw_name = known
                break
        if not matched_known:
            continue
        pre_label = None
        for key, label in SECTION_TO_LABEL.items():
            if key.lower() in raw_name.lower() or raw_name.lower() in key.lower():
                pre_label = label
                break
        boundaries.append({"pos": m.start(), "letter": m.group(1),
                           "section_name": raw_name, "pre_label": pre_label})
    deduped = []
    for b in boundaries:
        if deduped and abs(b["pos"] - deduped[-1]["pos"]) < 500:
            continue
        deduped.append(b)
    return deduped


def is_cunningham_text(text):
    third = len(re.findall(
        r'Buffett\s+(says|argues|believes|explains|notes|writes|instructs|emphasizes|observes|contends)',
        text, re.IGNORECASE))
    first = len(re.findall(r'\b(I |we |our |Charlie and I|my partner|Charlie Munger)', text))
    return third >= 2 and first == 0


def _sub_chunk(text, source_file, section, pre_label=None, max_chars=4000, overlap=500):
    if len(text) <= max_chars:
        return [Chunk(text=text, source_file=source_file,
                      source_section=section, chunk_strategy="thematic",
                      pre_label=pre_label)]
    paragraphs = [p.strip() for p in text.split('\n') if len(p.strip()) > 30]
    sub_chunks, current, current_len = [], [], 0
    for para in paragraphs:
        if current_len + len(para) > max_chars and current:
            sub_chunks.append(Chunk(
                text='\n'.join(current), source_file=source_file,
                source_section=section, chunk_strategy="thematic_subchunk",
                pre_label=pre_label))
            overlap_paras, overlap_len = [], 0
            for p in reversed(current):
                if overlap_len + len(p) > overlap: break
                overlap_paras.insert(0, p)
                overlap_len += len(p)
            current = overlap_paras + [para]
            current_len = overlap_len + len(para)
        else:
            current.append(para)
            current_len += len(para)
    if current:
        sub_chunks.append(Chunk(
            text='\n'.join(current), source_file=source_file,
            source_section=section,
            chunk_strategy="thematic_subchunk" if sub_chunks else "thematic",
            pre_label=pre_label))
    return sub_chunks


def chunk_cunningham_essays(pdf_path):
    raw = extract_essays_text(pdf_path)
    raw = re.sub(r'\d{4}\]\s*\n\s*THE ESSAYS OF WARREN BUFFETT\s*\n\s*\d+', '', raw)
    raw = re.sub(r'\d+\s*\n\s*CARDOZO LAW REVIEW\s*\n\s*\[Vol\.\s*\d+:\d+', '', raw)
    raw = re.sub(r'^\d{1,3}\s*$', '', raw, flags=re.MULTILINE)

    boundaries = find_section_boundaries(raw)
    all_chunks = []
    cunningham_filtered = 0

    for i, b in enumerate(boundaries):
        start = b["pos"]
        end = boundaries[i + 1]["pos"] if i + 1 < len(boundaries) else len(raw)
        section_text = raw[start:end].strip()
        if len(section_text) < 100:
            continue
        paragraphs = section_text.split('\n')
        clean = []
        for para in paragraphs:
            para = para.strip()
            if len(para) < 30: continue
            if is_cunningham_text(para):
                cunningham_filtered += 1
            else:
                clean.append(para)
        if not clean: continue
        clean_text = '\n'.join(clean)
        section_name = f"{b['letter']}. {b['section_name']}"
        all_chunks.extend(_sub_chunk(clean_text, "cunningham_essays.pdf",
                                     section_name, pre_label=b["pre_label"]))

    merged = []
    for chunk in all_chunks:
        if len(chunk.text) < 1000 and merged:
            merged[-1].text += '\n' + chunk.text
        else:
            merged.append(chunk)

    pre_labeled = sum(1 for c in merged if c.pre_label is not None)
    needs_llm = sum(1 for c in merged if c.pre_label is None)
    print(f"Sections: {len(boundaries)} | Cunningham filtered: {cunningham_filtered} | "
          f"Chunks: {len(merged)} | Pre-labeled: {pre_labeled} | Need LLM: {needs_llm}")
    return merged

In [3]:
chunks = chunk_cunningham_essays(
    Path("../sources/Essays-of-Warren-Buffett-_-Lessons-for-Corporate-America_Cunningham.pdf"))

Sections: 32 | Cunningham filtered: 0 | Chunks: 131 | Pre-labeled: 131 | Need LLM: 0


## Stage 2: Classification

Pre-labeled chunks inherit their label from the `SECTION_TO_LABEL` mapping and skip the LLM call entirely. Only chunks whose section names did not match a known mapping get sent to DeepSeek for classification. This reduces classification cost by approximately 85% while maintaining higher accuracy than LLM-only classification, since the labels are derived from the document's own thematic structure.

In [4]:
classified = await classify_chunks(chunks)
save_checkpoint(classified, "cunningham_classified")

Classifying 0 chunks (skipping 131 pre-labeled)

Label distribution:
  Strategy Development: 70
  Risk Management: 33
  Timing: 11
  Adaptability: 7
  Personal Life: 5
  Psychology: 5
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\cunningham_classified.pkl


In [5]:
dist = Counter(c.label for c in classified if c.label)
print("Label distribution:\n")
for label, count in dist.most_common():
    bar = "█" * count
    print(f"  {label:25s} {count:3d} {bar}")
failed = [c for c in classified if c.label is None]
if failed:
    print(f"\n!! {len(failed)} chunks unclassified")

Label distribution:

  Strategy Development       70 ██████████████████████████████████████████████████████████████████████
  Risk Management            33 █████████████████████████████████
  Timing                     11 ███████████
  Adaptability                7 ███████
  Personal Life               5 █████
  Psychology                  5 █████


## Stage 3: Q&A Generation

131 chunks × 3 candidate pairs each = approximately 393 raw pairs. Pre-assigned labels flow directly into the generation prompt, ensuring each chunk receives a label-specific prompt that enforces grounded answers, 3–5 sentence self-contained responses, and sublabel coverage — regardless of whether the label came from structural mapping or LLM classification.

In [6]:
pairs = await generate_all(classified, n_pairs=3)
save_checkpoint(pairs, "cunningham_pairs_raw")
print(f"\nTotal raw pairs: {len(pairs)}")

  5/131 chunks | 15 pairs
  10/131 chunks | 30 pairs
  15/131 chunks | 45 pairs
  20/131 chunks | 60 pairs
  25/131 chunks | 75 pairs
  30/131 chunks | 90 pairs
  35/131 chunks | 105 pairs
  40/131 chunks | 120 pairs
  45/131 chunks | 135 pairs
  50/131 chunks | 150 pairs
  55/131 chunks | 165 pairs
  60/131 chunks | 180 pairs
  65/131 chunks | 195 pairs
  70/131 chunks | 210 pairs
  75/131 chunks | 225 pairs
  80/131 chunks | 240 pairs
  85/131 chunks | 255 pairs
  90/131 chunks | 270 pairs
  95/131 chunks | 285 pairs
  100/131 chunks | 300 pairs
  105/131 chunks | 315 pairs
  110/131 chunks | 330 pairs
  115/131 chunks | 345 pairs
  120/131 chunks | 360 pairs
  125/131 chunks | 375 pairs
  130/131 chunks | 390 pairs
  131/131 chunks | 393 pairs
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\cunningham_pairs_raw.pkl

Total raw pairs: 393


In [7]:
seen_labels = set()
for p in pairs:
    if p.label in seen_labels: continue
    seen_labels.add(p.label)
    print(f"── {p.label} ──")
    print(f"  Q: {p.question}")
    print(f"  A: {p.answer[:250]}...")
    print()

── Strategy Development ──
  Q: How does Warren Buffett's concept of 'owner-orientation' shape the relationship between Berkshire Hathaway and its shareholders?
  A: Buffett views Berkshire's form as corporate but its attitude as a partnership. He and Charlie Munger think of shareholders as owner-partners and themselves as managing partners, seeing the company as a conduit for shareholder ownership of assets, not...

── Adaptability ──
  Q: How did Warren Buffett demonstrate adaptability in his approach to Berkshire Hathaway's original textile business?
  A: Buffett adapted by gradually shifting capital away from the struggling textile operation into more profitable ventures. In 1967, cash generated from textiles was used to fund the entry into insurance via the purchase of National Indemnity Company. He...

── Personal Life ──
  Q: How did Warren Buffett's personal values influence Berkshire Hathaway's policy on charitable donations?
  A: Buffett's belief that corporate charity should

## Stage 4: Quality Scoring & Filtering

Each candidate pair is scored by DeepSeek against its source chunk on four dimensions: groundedness (weight 0.35), label fit (0.25), richness (0.20), and novelty (0.20). The composite weighted average must exceed 0.70 for the pair to pass. The chunk map links each pair back to the exact text it was generated from, enabling the scorer to verify that all claims in the answer are directly supported by the source passage.

In [8]:
chunk_map = {c.chunk_id: c for c in classified}
scored = await score_all(pairs, chunk_map)
filtered = filter_by_quality(scored, threshold=0.7)
save_checkpoint(filtered, "cunningham_pairs_filtered")

  Scored 10/393
  Scored 20/393
  Scored 30/393
  Scored 40/393
  Scored 50/393
  Scored 60/393
  Scored 70/393
  Scored 80/393
  Scored 90/393
  Scored 100/393
  Scored 110/393
  Scored 120/393
  Scored 130/393
  Scored 140/393
  Scored 150/393
  Scored 160/393
  Scored 170/393
  Scored 180/393
  Scored 190/393
  Scored 200/393
  Scored 210/393
  Scored 220/393
  Scored 230/393
  Scored 240/393
  Scored 250/393
  Scored 260/393
  Scored 270/393
  Scored 280/393
  Scored 290/393
  Scored 300/393
  Scored 310/393
  Scored 320/393
  Scored 330/393
  Scored 340/393
  Scored 350/393
  Scored 360/393
  Scored 370/393
  Scored 380/393
  Scored 390/393
  Scored 393/393
Quality filter: 316/393 passed (threshold=0.7)
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\cunningham_pairs_filtered.pkl


In [9]:
scores = [p.composite_score for p in filtered if p.composite_score]
print(f"Pairs after filtering: {len(filtered)} / {len(pairs)} raw")
print(f"Drop rate: {(1 - len(filtered)/len(pairs))*100:.1f}%\n")
print(f"Score range: {min(scores):.2f} — {max(scores):.2f}")
print(f"Mean:   {statistics.mean(scores):.2f}")
print(f"Median: {statistics.median(scores):.2f}")

Pairs after filtering: 316 / 393 raw
Drop rate: 19.6%

Score range: 0.70 — 0.91
Mean:   0.82
Median: 0.83


## Stage 5: Coverage & Export

This document is expected to make significant contributions to Risk Management (junk bonds, zero-coupon bonds, preferred stock, leveraged buyouts), Timing (arbitrage, stock splits, intrinsic value), and Adaptability (plant closings, accounting shenanigans, tax burden). Strategy Development will accumulate additional pairs but has already exceeded the 100-pair target from the partnership letters.

Output files are written to `intermediate/` for merging in the assembly notebook.

In [10]:
print("Cunningham essays contribution:\n")
report = coverage_audit(filtered)

Cunningham essays contribution:

  Personal Life              13 pairs ( 4.1%)
    Sublabels hit: habits, personal_values, lifestyle
  Strategy Development      166 pairs (52.5%)
    Sublabels hit: value_investing_framework, margin_of_safety, competitive_moat, business_quality, circle_of_competence, capital_allocation, graham_influence
  Timing                     25 pairs ( 7.9%)
    Sublabels hit: entry_criteria, exit_criteria, market_valuation, patience, price_vs_value, market_cycles
  Risk Management            83 pairs (26.3%)
    Sublabels hit: position_sizing, diversification, leverage_avoidance, permanent_loss, insurance_float, margin_of_safety_risk, concentration
  Adaptability               16 pairs ( 5.1%)
    Sublabels hit: crisis_response, strategy_evolution, mistake_correction, market_regime_shifts, new_opportunities
  Psychology                 13 pairs ( 4.1%)
    Sublabels hit: temperament, emotional_discipline, contrarian_thinking, patience_psychology, fear_greed, ind

In [11]:
export_csv(filtered, Path("../intermediate/cunningham_qa.csv"))
export_detailed(filtered, Path("../intermediate/cunningham_qa_detailed.csv"))

Exported 316 pairs to ..\intermediate\cunningham_qa.csv
  Strategy Development: 166
  Risk Management: 83
  Timing: 25
  Adaptability: 16
  Personal Life: 13
  Psychology: 13
Detailed export: ..\intermediate\cunningham_qa_detailed.csv


In [12]:
label_counts = Counter(p.label for p in filtered)
print(f"FINAL: {len(filtered)} pairs across {len(label_counts)} labels\n")
for label, count in label_counts.most_common():
    best = max((p for p in filtered if p.label == label), key=lambda p: p.composite_score)
    print(f"── {label} ({count} pairs, best: {best.composite_score:.2f}) ──")
    print(f"  Q: {best.question}")
    print(f"  A: {best.answer[:300]}")
    print()

FINAL: 316 pairs across 6 labels

── Strategy Development (166 pairs, best: 0.91) ──
  Q: How did Warren Buffett's view on what constitutes a good investment evolve over time?
  A: Buffett states that it took him 20 years to recognize the importance of buying good businesses rather than just hunting for statistical 'bargains.' His early focus on cheap purchases led him into poor businesses like short-line farm implement manufacturers, third-place department stores, and a New E

── Risk Management (83 pairs, best: 0.90) ──
  Q: According to the passage, what flaw in the junk bond salesmen's logic does Warren Buffett identify, and what analogy does he use to illustrate it?
  A: Buffett identifies a statistical flaw where salesmen assumed new junk bonds were identical to older 'fallen angel' bonds, making their historical default data meaningless for prediction. He illustrates this error with the analogy of checking the historical death rate from Kool-Aid before drinking th

── Timing (25

In [13]:
# classified = load_checkpoint("cunningham_classified")
# pairs = load_checkpoint("cunningham_pairs_raw")
# filtered = load_checkpoint("cunningham_pairs_filtered")